<a href="https://colab.research.google.com/github/lewinskie254/stock-market-prediction/blob/main/predict_stock_prices_NeuralNets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
!pip install pytorch_forecasting pytorch_lightning

In [20]:
import numpy as np
import pandas as pd
import yfinance as yf
import torch
import torch.nn as nn
from tqdm.notebook import tqdm, trange
from statsmodels.tsa.statespace.sarimax import SARIMAX
from torch.utils.data import Dataset
from pytorch_forecasting.data.timeseries._timeseries import TimeSeriesDataSet
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_forecasting import Baseline, TemporalFusionTransformer, TimeSeriesDataSet
from pytorch_forecasting.data import GroupNormalizer
from pytorch_forecasting.metrics import SMAPE, PoissonLoss, QuantileLoss
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import optimize_hyperparameters

In [45]:
ticker = "AAPL"
stock = yf.Ticker(ticker)

# Get historical market data (e.g., past 1 year, daily interval)
hist = stock.history(period="1y", interval="1h")
print(hist.head())

# Get basic company info / metadata
info = stock.info
print(info.get("longName"), info.get("regularMarketPrice"))

# Get dividends and stock splits
dividends = stock.dividends
splits = stock.splits

                                 Open        High         Low       Close  \
Datetime                                                                    
2025-11-07 09:30:00-05:00  269.795013  272.290009  267.269989  271.579987   
2025-11-07 10:30:00-05:00  271.559998  271.970001  269.480011  269.529999   
2025-11-07 11:30:00-05:00  269.519989  269.890015  268.320007  269.109985   
2025-11-07 12:30:00-05:00  269.114990  269.140015  268.058807  268.450012   
2025-11-07 13:30:00-05:00  268.450012  268.660004  267.095001  267.250000   

                            Volume  Dividends  Stock Splits  
Datetime                                                     
2025-11-07 09:30:00-05:00  7855390        0.0           0.0  
2025-11-07 10:30:00-05:00  3991205        0.0           0.0  
2025-11-07 11:30:00-05:00  3276263        0.0           0.0  
2025-11-07 12:30:00-05:00  2041690        0.0           0.0  
2025-11-07 13:30:00-05:00  2217548        0.0           0.0  
Apple Inc. 342.37


In [19]:
hist.head()

,Open,High,Low,Close,Volume,Dividends,Stock Splits
Datetime,,,,,,,
2025-11-07 09:30:00-05:00,269.795013,272.290009,267.269989,271.579987,7855390,0.0,0.0
2025-11-07 10:30:00-05:00,271.559998,271.970001,269.480011,269.529999,3991205,0.0,0.0
2025-11-07 11:30:00-05:00,269.519989,269.890015,268.320007,269.109985,3276263,0.0,0.0
2025-11-07 12:30:00-05:00,269.114990,269.140015,268.058807,268.450012,2041690,0.0,0.0
2025-11-07 13:30:00-05:00,268.450012,268.660004,267.095001,267.250000,2217548,0.0,0.0


In [12]:
class DataLoading(Dataset):
  def __init__(self, data: pd.DataFrame, context_size: int, train_length: int, test_length: int, val_length) -> None:
    super().__init__()
    self.context_size = context_size
    self.train_length = train_length
    self.test_length = test_length
    self.val_length = val_length

  def __len__(self):
    return self.train_length + self.test_length + self.val_length

  def load_dataset():
    ...


  def __getitem__(self, index):
    ...


In [25]:
hist.index[0]

Timestamp('2025-11-07 09:30:00-0500', tz='America/New_York')

In [51]:
data = hist.drop(columns= ['Dividends', 'Stock Splits'])

In [52]:
data["year"] = data.index.year
data["month"] = data.index.month
data["day"] = data.index.dayofweek
data["date"] = data.index.day
data["hour"] = data.index.hour
data["cos_hour"] = np.cos(data.index.hour / 60 * 2 * np.pi)
data["sin_hour"] = np.sin(data.index.hour / 60 * 2 * np.pi)

In [53]:
data.tail()

,Open,High,Low,Close,Volume,year,month,day,date,hour,cos_hour,sin_hour
Datetime,,,,,,,,,,,,
2026-11-06 11:30:00-05:00,344.750000,344.989990,343.467194,344.119995,3181655,2026,11,4,6,11,4.067366e-01,0.913545
2026-11-06 12:30:00-05:00,344.100006,344.549896,343.179993,344.375000,2173497,2026,11,4,6,12,3.090170e-01,0.951057
2026-11-06 13:30:00-05:00,344.380005,344.869995,344.010010,344.170013,2101963,2026,11,4,6,13,2.079117e-01,0.978148
2026-11-06 14:30:00-05:00,344.149994,344.479797,342.829987,343.765015,2585934,2026,11,4,6,14,1.045285e-01,0.994522
2026-11-06 15:30:00-05:00,343.739990,344.250000,342.130005,342.364990,3380948,2026,11,4,6,15,6.123234e-17,1.000000


In [32]:
data["time_idx"] -= data["time_idx"].min()